This script converts CityGML LoD2 files into individual GeoPackages by extracting building footprints and attributes. Already processed files are skipped, and processing statistics are reported at the end.

In [ ]:
import os
import glob
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union
import xml.etree.ElementTree as ET
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')


class CityGMLParser:
    """Parser for CityGML LoD2 files"""
    
    def __init__(self):
        self.namespaces = {
            'gml': 'http://www.opengis.net/gml',
            'bldg': 'http://www.opengis.net/citygml/building/1.0',
            'gen': 'http://www.opengis.net/citygml/generics/1.0',
            'citygml': 'http://www.opengis.net/citygml/1.0'
        }
    
    def parse_poslist_to_polygon(self, poslist_text):
        """Converts a GML posList into a 2D Shapely polygon (X, Y coordinates only)"""
        try:
            coords = list(map(float, poslist_text.split()))
            points = [(coords[i], coords[i+1]) for i in range(0, len(coords), 3)]
            
            if len(points) >= 3:
                return Polygon(points)
            return None
        except:
            return None
    
    def get_building_footprint(self, building_elem):
        """Extracts the 2D building footprint from the GroundSurface"""
        ground_polygons = []

        # Use only the GroundSurface, which represents the exact building footprint
        for surface in building_elem.findall('.//bldg:GroundSurface', self.namespaces):
            poslist = surface.find('.//gml:posList', self.namespaces)
            if poslist is not None and poslist.text:
                poly = self.parse_poslist_to_polygon(poslist.text)
                if poly and poly.is_valid:
                    ground_polygons.append(poly)

        if ground_polygons:
            try:
                # Merge multiple GroundSurfaces (e.g., buildings with courtyards) using a union operation
                return unary_union(ground_polygons)
            except:
                return ground_polygons[0]

        # Fallback: if no GroundSurface is available, use the WallSurface instead
        wall_polygons = []
        for surface in building_elem.findall('.//bldg:WallSurface', self.namespaces):
            poslist = surface.find('.//gml:posList', self.namespaces)
            if poslist is not None and poslist.text:
                poly = self.parse_poslist_to_polygon(poslist.text)
                if poly and poly.is_valid:
                    wall_polygons.append(poly)

        if wall_polygons:
            try:
                return unary_union(wall_polygons)
            except:
                return None

        return None
    
    def extract_attributes(self, building_elem):
        """Extracts all available building attributes"""
        attributes = {}
        
        gml_id = building_elem.get('{http://www.opengis.net/gml}id')
        attributes['gml_id'] = gml_id
        
        creation_date = building_elem.find('citygml:creationDate', self.namespaces)
        if creation_date is not None:
            attributes['creation_date'] = creation_date.text
        
        ext_ref = building_elem.find('.//citygml:externalObject/citygml:name', self.namespaces)
        if ext_ref is not None:
            attributes['external_ref'] = ext_ref.text
        
        for gen_attr in building_elem.findall('.//gen:stringAttribute', self.namespaces):
            attr_name = gen_attr.get('name')
            attr_value = gen_attr.find('gen:value', self.namespaces)
            if attr_name and attr_value is not None:
                try:
                    if '.' in attr_value.text:
                        attributes[attr_name] = float(attr_value.text)
                    else:
                        attributes[attr_name] = attr_value.text
                except:
                    attributes[attr_name] = attr_value.text
        
        function = building_elem.find('bldg:function', self.namespaces)
        if function is not None:
            attributes['function'] = function.text
        
        roof_type = building_elem.find('bldg:roofType', self.namespaces)
        if roof_type is not None:
            attributes['roof_type'] = roof_type.text
        
        measured_height = building_elem.find('bldg:measuredHeight', self.namespaces)
        if measured_height is not None:
            try:
                attributes['measured_height'] = float(measured_height.text)
            except:
                attributes['measured_height'] = measured_height.text
        
        storeys = building_elem.find('bldg:storeysAboveGround', self.namespaces)
        if storeys is not None:
            try:
                attributes['storeys_above_ground'] = int(storeys.text)
            except:
                attributes['storeys_above_ground'] = storeys.text
        
        return attributes


def parse_single_gml_file(gml_filepath, output_dir, parser):
    """
    Parses a single GML file and saves it as a GeoPackage
    """
    filename = os.path.basename(gml_filepath)
    filename_without_ext = os.path.splitext(filename)[0]
    
    buildings = []
    
    try:
        print(f"Parsing: {filename}...", end=" ")
        
        tree = ET.parse(gml_filepath)
        root = tree.getroot()
        
        for building in root.findall('.//bldg:Building', parser.namespaces):
            footprint = parser.get_building_footprint(building)
            
            if footprint and footprint.is_valid:
                attrs = parser.extract_attributes(building)
                attrs['geometry'] = footprint
                attrs['source_file'] = filename
                buildings.append(attrs)
        
        if buildings:
            # Als GeoPackage speichern
            gpkg_filepath = os.path.join(output_dir, f'{filename_without_ext}.gpkg')
            gdf = gpd.GeoDataFrame(buildings, crs='EPSG:25832')
            gdf.to_file(gpkg_filepath, layer='buildings', driver='GPKG')
            
            file_size_kb = os.path.getsize(gpkg_filepath) / 1024
            print(f"✓ {len(buildings)} Gebäude → {file_size_kb:.1f} KB")
            
            return gpkg_filepath, len(buildings)
        else:
            print(f"⚠ No buildings found")
            return None, 0
    
    except Exception as e:
        print(f"✗ ERROR: {str(e)[:60]}")
        return None, 0


def process_gml_files_simple(input_dir, output_dir, start_file=1, end_file=2000):
    """
    Processes GML files individually    
    """
    
    start_time = datetime.now()
    
    # Alle GML-Dateien finden
    all_gml_files = sorted(glob.glob(os.path.join(input_dir, '*.gml')))
    
    # Slice für gewünschten Bereich (1-basiert zu 0-basiert)
    gml_files = all_gml_files[start_file-1:end_file]
    total_files = len(gml_files)
    
    if total_files == 0:
        print("No GLM File found!")
        return
    
    # Output-Verzeichnis erstellen
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output-Verzeichnis: {output_dir}")
    
    # Prüfe welche Dateien bereits existieren
    print(f"\nPrüfe auf bereits existierende GeoPackages...")
    files_to_process = []
    files_already_exist = []
    
    for gml_file in gml_files:
        filename_without_ext = os.path.splitext(os.path.basename(gml_file))[0]
        gpkg_filepath = os.path.join(output_dir, f'{filename_without_ext}.gpkg')
        
        if os.path.exists(gpkg_filepath):
            files_already_exist.append(gml_file)
        else:
            files_to_process.append(gml_file)
    
    
    if len(files_to_process) == 0:
        print("\n✓ All files have already been processed")
        return
    
    print(f"\n{'='*70}")
    print(f"Verarbeite {len(files_to_process)} new GLM-files")
    print(f"{'='*70}\n")
    
    # Initialize the parser
    parser = CityGMLParser()
    
    # Statistics
    created_gpkgs = []
    total_buildings = 0
    files_with_buildings = 0
    files_empty = 0
    files_error = 0
    
    # Process each file individually
    for idx, gml_file in enumerate(files_to_process, 1):

        abs_idx = all_gml_files.index(gml_file) + 1
        
        print(f"[{idx:4d}/{len(files_to_process)}] (File #{abs_idx:4d}) ", end="")
        
        gpkg_path, building_count = parse_single_gml_file(gml_file, output_dir, parser)
        
        if gpkg_path:
            created_gpkgs.append(gpkg_path)
            total_buildings += building_count
            files_with_buildings += 1
        elif building_count == 0:
            files_empty += 1
        else:
            files_error += 1
    
    # Final statistics
    end_time = datetime.now()
    duration = end_time - start_time
    
    
    def main():
    """main"""
    
    # ========================
    # CONFIGURATION – MODIFY THESE SETTINGS AS REQUIRED
    # ========================
    
    input_directory = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\LoD2\download"
    output_directory = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\LoD2\download\individual_gpkgs"
    
    # Range of files to process (1-based indexing)
    start_file_number = 1
    end_file_number = 18300
    
    # ========================
    # VERARBEITUNG
    # ========================
    if not os.path.exists(input_directory):
        print(f"ERROR: Input-directory doesn't exist!")
        return
    
    process_gml_files_simple(
        input_directory, 
        output_directory,
        start_file=start_file_number,
        end_file=end_file_number
    )
    
    print(f"\End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


if __name__ == "__main__":
    main()
